# Portfolio Optimization 4.1 — Adding a Risk-Free Asset
### From short selling to Tobin's separation theorem

**Previously:** we optimized portfolios of risky assets using Markowitz's framework, including **short positions and gross-exposure limits**. We saw that short selling can improve investment opportunities, but may also amplify risk.

**Today's question:** What changes if, in addition to the same risky assets, investors can **lend or borrow at a known risk-free rate**?

By the end, you should be able to:
1. Interpret a negative *risk-free-asset weight* as borrowing, not as a short sale of a stock.
2. Calculate the return and volatility of a mix of a risky portfolio and a risk-free asset.
3. Explain why the best risky portfolio is the **maximum-Sharpe (tangency) portfolio** under the model's assumptions.
4. Connect this result to your earlier short-selling constraints.
5. Build portfolios for investors with different risk appetites without re-optimizing their risky-asset composition.

> **Teaching model:** the risk-free rate is assumed known for the investment horizon, and lending and borrowing occur at the **same rate**. Real borrowing rates, collateral, fees, taxes and short-selling costs may differ. We return to these assumptions at the end.

## 1. A new investment decision: lend, invest, or borrow?

Suppose you have **€100** and a risky asset that you expect to earn more than a risk-free investment.

- **Conservative choice:** invest €40 in the risky asset and **lend** €60 at the risk-free rate.
- **Fully invested:** invest the entire €100 in the risky asset.
- **Leveraged choice:** invest €140 in the risky asset by using €100 of capital and **borrowing** €40 at the risk-free rate.

A negative weight in the **risk-free asset** means **borrowing**. It is different from a negative weight in a stock, which represents **short exposure to that stock**.

**Think first:** If the risky asset falls 10%, what happens to each of these three portfolios? Which one is most exposed to that fall?

### The two-asset formula

Let $y$ denote the weight in the **risky portfolio** and $1-y$ the weight in the **risk-free asset**.

$$E[R_p]=y\mu_R+(1-y)r_f=r_f+y(\mu_R-r_f)$$

$$\sigma_p=|y|\sigma_R$$

The risk-free asset has zero return variance and zero covariance with the risky portfolio **by assumption over this horizon**. Therefore, the combinations are a **straight line** in expected-return/volatility space when $y\geq0$.

| Risky weight $y$ | Risk-free weight $1-y$ | Economic meaning |
|---:|---:|---|
| 0.40 | 0.60 | Invest 40% in risky assets, lend 60% |
| 1.00 | 0.00 | Fully invested in risky assets |
| 1.40 | −0.40 | Borrow 40% and invest 140% in risky assets |

The **absolute value** in the volatility formula is essential. A standard deviation cannot be negative.

## 2. Load historical prices and define consistent annual inputs

We reuse the **2015–2019** example and the same stock universe as in the previous notebook. Unlike the earlier log-return implementation, we use **simple returns**, because portfolio returns are weighted sums of simple asset returns. All optimization inputs and charts are expressed **annually**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp

# Colab / Jupyter: install first if necessary: %pip install yfinance cvxpy
try:
    import yfinance as yf
except ImportError as exc:
    raise ImportError("Install yfinance using %pip install yfinance") from exc

TRADING_DAYS = 252
RF = 0.03                   # Annual risk-free rate, 3%
START, END = "2015-01-01", "2019-12-30"
ASSETS = [
    'BA', 'BAX', 'BMY', 'CMCSA', 'CNP', 'CPB', 'DE', 'F', 'GE', 'HPQ',
    'JCI', 'JPM', 'KHC', 'LUV', 'MMC', 'MO', 'NI', 'PCAR', 'PSA', 'SEE',
    'T', 'TGT', 'TMO', 'TXT', 'VZ', 'ZION'
]
MAX_MISSING = 0.10
LOCAL_PRICE_CSV = None  # Optional: path to an existing wide CSV (Date, BA, GE, ...)

print("CVXPY:", cp.__version__)
print("Installed solvers:", cp.installed_solvers())

In [ ]:
if LOCAL_PRICE_CSV:
    prices_raw = pd.read_csv(LOCAL_PRICE_CSV, index_col=0, parse_dates=True)
else:
    download = yf.download(
        ASSETS, start=START, end=END,
        auto_adjust=True, progress=False, threads=False, group_by="column"
    )
    if download.empty:
        raise RuntimeError("Yahoo Finance returned no data. Try again or set LOCAL_PRICE_CSV.")
    # With auto_adjust=True, 'Close' contains prices adjusted for splits/dividends.
    prices_raw = download['Close'] if isinstance(download.columns, pd.MultiIndex) else download[['Close']]
    if not isinstance(download.columns, pd.MultiIndex):
        prices_raw.columns = [ASSETS[0]]

# Preserve actual ticker labels; never overwrite column names with the original list.
prices_raw = prices_raw.sort_index().apply(pd.to_numeric, errors="coerce")
missing_share = prices_raw.isna().mean().sort_values(ascending=False)
excluded = missing_share[missing_share > MAX_MISSING].index.tolist()
prices = prices_raw.loc[:, missing_share[missing_share <= MAX_MISSING].index].copy()
prices = prices.dropna(axis=0, how="any")   # No forward fill; common observed dates
assets = prices.columns.tolist()             # Exactly aligned with return and covariance data

print("Failed/excluded tickers:", sorted(set(ASSETS) - set(assets)))
print("Observations after alignment:", len(prices))
print("Assets remaining:", len(assets))
if len(assets) < 3 or len(prices) < 252:
    raise ValueError("Insufficient common observations or assets. Check the download / CSV.")

returns = prices.pct_change(fill_method=None).dropna()
mu = (returns.mean() * TRADING_DAYS).to_numpy()             # Annualized arithmetic mean
Sigma = (returns.cov() * TRADING_DAYS).to_numpy()          # Annualized covariance
# Tiny numerical stabilization only (not an estimation or shrinkage method).
Sigma = 0.5 * (Sigma + Sigma.T)
Sigma += 1e-12 * np.eye(len(assets))
N = len(assets)
summary = pd.DataFrame({"Expected return (annual)": mu,
                        "Volatility (annual)": np.sqrt(np.diag(Sigma))}, index=assets)
display(summary.sort_values("Expected return (annual)", ascending=False).round(4))

**Data checks.** Some historical tickers may now be unavailable through Yahoo Finance. We identify failed downloads, exclude assets with more than 10% missing adjusted prices, and require a common sample. **Do not silently relabel columns:** an asset's name must always match its return series and covariance row. The resulting sample may differ from the original historical notebook. For reproducible assessment, distribute a fixed course CSV.

The expected returns and covariances are **historical estimates**, not promises about future returns or risks. Annualization uses 252 trading days and arithmetic daily returns; the risk-free rate is a **hypothetical constant annual rate**, not a historical daily series.

## 3. One risky stock + the risk-free asset

In [ ]:
# Prefer the original teaching example (BA) if its history is available.
RISKY_TICKER = 'BA' if 'BA' in assets else summary['Expected return (annual)'].idxmax()
mu_R = float(summary.loc[RISKY_TICKER, 'Expected return (annual)'])
sigma_R = float(summary.loc[RISKY_TICKER, 'Volatility (annual)'])

print(f"Risky asset: {RISKY_TICKER}")
print(f"Annual expected return: {mu_R:.2%}")
print(f"Annual volatility: {sigma_R:.2%}")
print(f"Risk-free lending/borrowing rate: {RF:.2%}")

weights_R = np.linspace(0, 1.6, 200)
mixed_mu = RF + weights_R * (mu_R - RF)
mixed_sigma = np.abs(weights_R) * sigma_R

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(mixed_sigma, mixed_mu, label='Risk-free + risky asset')
ax.scatter([0, sigma_R], [RF, mu_R], s=70, zorder=4)
ax.annotate('Risk-free asset', (0, RF), xytext=(6, 8), textcoords='offset points')
ax.annotate(RISKY_TICKER, (sigma_R, mu_R), xytext=(6, 8), textcoords='offset points')
for y, name in [(0.4, 'Lending'), (1.4, 'Borrowing')]:
    ax.scatter([y * sigma_R], [RF + y*(mu_R - RF)], s=65, zorder=5)
    ax.annotate(f'{name}: {y:.0%} risky', (y*sigma_R, RF+y*(mu_R-RF)),
                xytext=(7, -16), textcoords='offset points')
ax.set(xlabel='Annual volatility', ylabel='Annual expected return',
       title='Lending and borrowing against one risky asset')
ax.grid(alpha=0.25)
ax.legend()
plt.show()

### Exercise 1 — Three portfolio decisions

Using the estimated expected return and volatility of the risky stock above, calculate and interpret:

**A.** The risky and risk-free weights needed for **14% annual expected return**.

**B.** The weights needed for **25% annual expected return**. Is borrowing required? Under what condition?

**C.** The weights needed for **7% annual volatility**, assuming a nonnegative risky weight.

**Predict first:** In which exercise is the risk-free weight negative? What happens if the risky asset has a lower expected return than the target?

In [ ]:
# Solution: mathematical results are based on downloaded estimates, not hardcoded returns.
def mix_for_target_return(target, risky_mu, risk_free):
    spread = risky_mu - risk_free
    if np.isclose(spread, 0):
        raise ValueError("The risky asset and risk-free asset have the same expected return.")
    return (target - risk_free) / spread   # weight in risky asset

def mix_for_target_volatility(target_sigma, risky_sigma):
    if risky_sigma <= 0:
        raise ValueError("Risky portfolio must have positive volatility.")
    return target_sigma / risky_sigma    # chooses the nonnegative-risky-weight branch

exercise_rows = []
for name, y in [
    ('A: target return 14%', mix_for_target_return(0.14, mu_R, RF)),
    ('B: target return 25%', mix_for_target_return(0.25, mu_R, RF)),
    ('C: target volatility 7%', mix_for_target_volatility(0.07, sigma_R))
]:
    exercise_rows.append({
        'Exercise': name, 'Risky weight': y, 'Risk-free weight': 1-y,
        'Expected return': RF+y*(mu_R-RF), 'Volatility': abs(y)*sigma_R,
        'Interpretation': 'Borrowing' if y>1 else ('Lending' if y<1 else 'Fully risky')
    })
exercise_df = pd.DataFrame(exercise_rows).set_index('Exercise')
display(exercise_df.style.format({
    'Risky weight': '{:.1%}', 'Risk-free weight': '{:.1%}',
    'Expected return': '{:.2%}', 'Volatility': '{:.2%}'
}))

**Discussion.** Borrowing is implied by $y>1$, which means $1-y<0$. A high expected return target does **not** automatically imply borrowing: the answer depends on the estimated risky-asset return. The target return might also imply a **short position in the risky asset** ($y<0$); the linear financing interpretation still applies, but that is a different investment decision.

So far, we have combined the risk-free asset with **one stock**. The stock need not be the best risky asset or portfolio. Which *portfolio* should we choose instead?

## 4. Recall Markowitz: risky portfolios with and without shorts

We already know how to construct the efficient frontier of **risky assets only**. Here we compare:

- **Long-only:** $\sum_i w_i=1$, $w_i\geq0$.
- **Short-selling permitted:** $\sum_i w_i=1$, $\sum_i|w_i|\leq1.5$.

The second constraint allows gross exposure up to **150%** of the capital allocated to the risky portfolio. At the limit it corresponds to up to 125% long and 25% short. This is the same gross-exposure idea from our previous notebook.

**Key distinction:** this 150% cap applies to the **composition of the risky sleeve**. Later, we separately choose how much total wealth is invested in that sleeve by borrowing/lending at $r_f$.

In [ ]:
# Reusable, explicit and solver-checked convex optimization functions.
AVAILABLE = cp.installed_solvers()
SOLVERS = [s for s in ('CLARABEL', 'SCS') if s in AVAILABLE]
if not SOLVERS:
    raise RuntimeError("Please install CLARABEL or SCS: %pip install clarabel scs")

def solve_checked(problem):
    errors = []
    for solver in SOLVERS:
        try:
            val = problem.solve(solver=solver, warm_start=True)
            if problem.status in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
                return val
            errors.append(f'{solver}: {problem.status}')
        except cp.error.SolverError as exc:
            errors.append(f'{solver}: {exc}')
    raise RuntimeError('Optimization failed: ' + '; '.join(errors))

def portfolio_stats(weights):
    w = np.asarray(weights, dtype=float).reshape(-1)
    return float(mu @ w), float(np.sqrt(max(0.0, w @ Sigma @ w)))

def risky_constraints(w, gross_limit):
    constraints = [cp.sum(w) == 1]
    if gross_limit == 1.0:
        constraints.append(w >= 0)
    else:
        constraints.append(cp.norm1(w) <= gross_limit)
    return constraints

def minimum_variance(gross_limit, target_return=None):
    w = cp.Variable(N)
    cons = risky_constraints(w, gross_limit)
    if target_return is not None:
        cons.append(mu @ w >= target_return)
    prob = cp.Problem(cp.Minimize(cp.quad_form(w, cp.psd_wrap(Sigma))), cons)
    solve_checked(prob)
    return np.asarray(w.value).reshape(-1)

def maximum_risky_return(gross_limit):
    # Here constraints are symmetric except for budget; the LP is convex.
    w = cp.Variable(N)
    prob = cp.Problem(cp.Maximize(mu @ w), risky_constraints(w, gross_limit))
    solve_checked(prob)
    return float(mu @ w.value)

def efficient_frontier(gross_limit, count=24):
    w_gmv = minimum_variance(gross_limit)
    lo = float(mu @ w_gmv)
    hi = maximum_risky_return(gross_limit)
    targets = np.linspace(lo, hi - max(1e-7, 1e-5*abs(hi-lo)), count)
    points = []
    for target in targets:
        w = minimum_variance(gross_limit, target)
        ret, vol = portfolio_stats(w)
        points.append((vol, ret))
    return np.array(points), w_gmv

frontiers = {}
for label, gross in [('Long-only', 1.0), ('Shorts allowed (gross <= 150%)', 1.5)]:
    pts, gmv = efficient_frontier(gross)
    frontiers[label] = {'points': pts, 'gmv': gmv, 'gross': gross}
    r, s = portfolio_stats(gmv)
    print(f'{label}: GMV return={r:.2%}, GMV volatility={s:.2%}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.sqrt(np.diag(Sigma)), mu, alpha=.35, s=22, label='Individual stocks')
for label, result in frontiers.items():
    pts = result['points']
    ax.plot(pts[:,0], pts[:,1], lw=2, label=label)
ax.scatter([0], [RF], marker='*', s=150, label='Risk-free asset')
ax.set(xlabel='Annual volatility', ylabel='Annual expected return',
       title='Markowitz risky-only efficient frontiers')
ax.grid(alpha=.25)
ax.legend()
plt.show()

**What did short selling change?** It expanded the risky-only feasible set, though the minimum-variance portfolio and the efficient frontier can move by different amounts. More freedom does not imply that every optimal solution must use short positions.

**New question:** If the investor is also allowed to lend at $r_f$, is it equally attractive to combine the risk-free asset with *every* efficient risky portfolio?

## 5. Compare straight lines from the risk-free asset

For a candidate risky portfolio with $(\mu_R,\sigma_R)$, all nonnegative-risky-weight combinations have the same slope:

$$\frac{E[R_p]-r_f}{\sigma_p}=\frac{\mu_R-r_f}{\sigma_R}$$

That slope is its **ex-ante Sharpe ratio**. A **steeper** line gives higher expected excess return for the same volatility, provided the slope is positive and lending/borrowing is available at the assumed rate.

We first plot lines through three risky portfolios. **Predict** which one dominates before inspecting the figure.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
chosen = frontiers['Long-only']['points'][[4, 11, 20], :]
y_grid = np.linspace(0, 1.5, 150)
for k, (vol, ret) in enumerate(chosen, start=1):
    ax.plot(y_grid*vol, RF+y_grid*(ret-RF), lw=1.8,
            label=f'Portfolio {k}: Sharpe {(ret-RF)/vol:.2f}')
    ax.scatter([vol], [ret], s=50)
pts = frontiers['Long-only']['points']
ax.plot(pts[:,0], pts[:,1], 'k--', alpha=.7, label='Long-only risky frontier')
ax.scatter([0], [RF], s=110, marker='*', label='Risk-free rate')
ax.set(xlabel='Annual volatility', ylabel='Annual expected return',
       title='Different risky portfolios: different capital-allocation lines')
ax.grid(alpha=.25)
ax.legend()
plt.show()

## 6. Which risky portfolio is best? The maximum-Sharpe portfolio

We now look for the risky portfolio with the highest positive expected excess return **per unit of volatility**:

$$\max_w \ \frac{(\mu-r_f\mathbf{1})^\top w}{\sqrt{w^\top\Sigma w}}$$

subject to the **risky-sleeve** constraints (long-only, or short selling with limited gross exposure).

This is not a valid CVXPY DCP expression to maximize directly. Instead, introduce scaled holdings $x$ and solve the equivalent convex problem (when positive excess-return portfolios exist):

$$\min_x\ x^\top\Sigma x \quad\text{subject to}\quad(\mu-r_f\mathbf{1})^\top x=1.$$

Then normalize: $w=x/(\mathbf{1}^\top x)$. For long-only, require $x\geq0$. For gross exposure $G$, add $\|x\|_1\leq G\mathbf{1}^\top x$. This keeps the **same normalized risky-sleeve gross limit**.

**Why this works:** fixing the expected excess return to one makes minimizing variance equivalent to maximizing the Sharpe ratio. Scaling $x$ does not change that ratio. We verify the resulting weights numerically.

In [ ]:
def tangency_portfolio(gross_limit):
    excess = mu - RF
    x = cp.Variable(N)
    cons = [excess @ x == 1]
    if gross_limit == 1.0:
        cons += [x >= 0]
    else:
        cons += [cp.sum(x) >= 1e-9, cp.norm1(x) <= gross_limit * cp.sum(x)]
    problem = cp.Problem(cp.Minimize(cp.quad_form(x, cp.psd_wrap(Sigma))), cons)
    solve_checked(problem)
    xval = np.asarray(x.value).reshape(-1)
    if xval.sum() <= 1e-10:
        raise ValueError('Tangency normalization failed: sum of x is not positive.')
    w = xval / xval.sum()
    expected, volatility = portfolio_stats(w)
    if expected <= RF or volatility <= 0:
        raise ValueError('No valid positive-excess-return tangency portfolio.')
    if gross_limit == 1.0 and w.min() < -1e-5:
        raise AssertionError('Invalid long-only weights.')
    if np.sum(np.abs(w)) > gross_limit + 1e-4:
        raise AssertionError('Gross-exposure limit violated.')
    return w, expected, volatility, (expected-RF)/volatility

tangencies = {}
for label, result in frontiers.items():
    w, ret, vol, sharpe = tangency_portfolio(result['gross'])
    tangencies[label] = {'w': w, 'ret': ret, 'vol': vol, 'sharpe': sharpe}

comparison = pd.DataFrame({label: {
    'Expected return': z['ret'], 'Volatility': z['vol'],
    'Ex-ante Sharpe': z['sharpe'], 'Gross exposure': sum(abs(z['w'])),
    'Number of short positions': int((z['w'] < -1e-5).sum())
} for label, z in tangencies.items()}).T
display(comparison.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for label, front in frontiers.items():
    pts = front['points']
    ax.plot(pts[:,0], pts[:,1], '--', lw=1.6, alpha=.7, label=f'{label}: risky frontier')
    z = tangencies[label]
    y = np.linspace(0, 1.6, 100)
    ax.plot(y*z['vol'], RF+y*(z['ret']-RF), lw=2.4,
            label=f'{label}: tangency line (Sharpe={z["sharpe"]:.2f})')
    ax.scatter([z['vol']], [z['ret']], s=75, zorder=5)
ax.scatter([0], [RF], s=140, marker='*', label='Risk-free asset')
ax.set(xlabel='Annual volatility', ylabel='Annual expected return',
       title='Tangency portfolios and their capital-allocation lines')
ax.grid(alpha=.25)
ax.legend(fontsize=8)
plt.show()

**Interpret carefully.** The maximum-Sharpe risky portfolio can differ when short selling is permitted. If the short-selling tangency line is steeper, it offers a better **estimated** excess return per unit of risk under the chosen risky-sleeve constraints. It does **not** guarantee superior future performance.

We should not assume the tangency portfolio contains shorts simply because shorts are permitted. Inspect the calculated weights.

In [ ]:
weights_table = pd.DataFrame({name: tangencies[name]['w'] for name in tangencies}, index=assets)
display(weights_table.loc[(weights_table.abs() > 0.001).any(axis=1)].round(3))

ax = weights_table.plot(kind='bar', figsize=(12, 4.5), width=.75)
ax.axhline(0, color='black', lw=.8)
ax.set(ylabel='Weight in risky sleeve', title='Tangency portfolio weights: two investment mandates')
ax.legend(fontsize=9)
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

## 7. Tobin's separation theorem: two decisions, not one

Under the **same expected returns, covariances, borrowing/lending rate, and risky-portfolio mandate**, all mean-variance investors can use the **same tangency risky portfolio**.

1. **Portfolio choice:** choose the risky tangency portfolio (its composition is determined by investment opportunities and constraints).
2. **Financing/risk choice:** select $y$, the amount of wealth invested in that portfolio; put $1-y$ in the risk-free asset.

A cautious investor might select $y=0.4$ and lend 60%. An aggressive investor might select $y=1.4$ and borrow 40%. The relative weights **inside the risky sleeve** do not change.

**Important nuance from the previous class:** a gross-exposure limit on the **whole portfolio** can constrain borrowing and change this conclusion. In today's teaching model, the gross-exposure cap applies only to the *normalized risky sleeve*, while lending/borrowing is governed separately.

In [ ]:
# Use the long-only tangency for the simplest explanation; the same calculation
# works with the shorts-allowed tangency by changing the dictionary key.
SELECTED_MANDATE = 'Long-only'
T = tangencies[SELECTED_MANDATE]

investor_rows = []
for name, y in [('Conservative', 0.40), ('Fully risky', 1.00), ('Aggressive', 1.40)]:
    investor_rows.append({
        'Investor': name, 'Risky sleeve': y, 'Risk-free asset': 1-y,
        'Expected return': RF+y*(T['ret']-RF), 'Volatility': abs(y)*T['vol'],
        'Total individual-stock weights sum': y
    })
investors = pd.DataFrame(investor_rows).set_index('Investor')
display(investors.style.format({
    'Risky sleeve': '{:.0%}', 'Risk-free asset': '{:.0%}',
    'Expected return': '{:.2%}', 'Volatility': '{:.2%}',
    'Total individual-stock weights sum': '{:.0%}'
}))

fig, ax = plt.subplots(figsize=(9,5.5))
y_grid = np.linspace(0, 1.6, 150)
ax.plot(y_grid*T['vol'], RF+y_grid*(T['ret']-RF), label='Capital allocation line')
for row_name, row in investors.iterrows():
    ax.scatter([row['Volatility']], [row['Expected return']], s=65, zorder=4)
    ax.annotate(row_name, (row['Volatility'], row['Expected return']),
                xytext=(8,5), textcoords='offset points')
ax.scatter([T['vol']], [T['ret']], marker='D', s=60, label='Same risky tangency portfolio')
ax.set(xlabel='Annual volatility', ylabel='Annual expected return',
       title='Different investors, one risky portfolio')
ax.legend(); ax.grid(alpha=.25)
plt.show()

### Exercise 2 — Target an annual expected return of 10%

An investor wants **10% expected annual return**. You have already calculated the risky tangency portfolio, and the risk-free rate is 3%.

**A.** What fraction $y$ should be allocated to the risky sleeve?

**B.** What proportion should be held in the risk-free asset?

**C.** What is the expected annual volatility?

**D.** Compare its volatility with a risky-only portfolio that must attain **exactly** 10% expected return under the **same risky-sleeve mandate**. Is the mixed portfolio more attractive?

Use the estimated inputs in the notebook, not assumed returns from another period.

In [ ]:
TARGET_RETURN = 0.10
y = mix_for_target_return(TARGET_RETURN, T['ret'], RF)
final_mu = RF + y*(T['ret'] - RF)
final_vol = abs(y)*T['vol']
print(f'Mandate: {SELECTED_MANDATE}')
print(f'Investment in risky tangency sleeve: {y:.2%}')
print(f'Investment in risk-free asset: {1-y:.2%}')
print(f'Expected return: {final_mu:.2%}')
print(f'Annual volatility: {final_vol:.2%}')
print('Interpretation:', 'borrowing' if y>1 else ('lending' if 0<=y<1 else 'shorting the risky sleeve'))

In [ ]:
# Same target return using risky assets only; equality enforces a fair comparison.
try:
    w_risky_only = cp.Variable(N)
    problem = cp.Problem(
        cp.Minimize(cp.quad_form(w_risky_only, cp.psd_wrap(Sigma))),
        risky_constraints(w_risky_only, frontiers[SELECTED_MANDATE]['gross']) +
        [mu @ w_risky_only == TARGET_RETURN]
    )
    solve_checked(problem)
    risky_only_mu, risky_only_vol = portfolio_stats(w_risky_only.value)
    print(f'Risky-only portfolio at {risky_only_mu:.2%} expected return: {risky_only_vol:.2%} volatility')
    print(f'Risk-free + tangency portfolio at {final_mu:.2%} expected return: {final_vol:.2%} volatility')
    if final_vol <= risky_only_vol + 1e-5:
        print('The mixed portfolio has lower or approximately equal estimated volatility.')
    else:
        print('Check constraints, inputs, and numerical precision: this comparison is unexpected.')
except RuntimeError as exc:
    print('Risky-only exact-return comparison could not be solved:', exc)

## 8. Conclusions and real-world caveats

**What changed compared with Markowitz and short selling?**

- Without the risk-free asset, investors choose among **risky-only** efficient portfolios.
- With a risk-free asset and symmetric borrowing/lending, the combinations of any risky portfolio form a **straight line** in expected return–volatility space (for a nonnegative risky allocation).
- The steepest feasible line is generated by the **maximum-Sharpe risky portfolio**.
- **Tobin's separation:** investors who share the same inputs and mandate can choose the same risky portfolio but differ in their allocation between that portfolio and the risk-free asset.
- **Short selling** changes the feasible set of risky portfolios and can change the tangency portfolio; **borrowing at the risk-free rate** changes the total size of the risky investment. These are related but distinct decisions.

**Assumptions and limitations to discuss:**

1. The risk-free return is defined for a specific horizon and currency; even government securities may carry mark-to-market risk if sold early.
2. In practice, borrowing can cost more than lending, and leverage involves margin, financing costs and constraints. The straight line may kink or cease to be feasible.
3. Historical estimates of $\mu$ and $\Sigma$ are uncertain. Maximum-Sharpe portfolios can be unstable, especially when short selling is permitted.
4. The short-selling gross cap was imposed on the **risky sleeve**, not automatically on the entire levered portfolio; an aggregate cap must be modeled separately.
5. This is an **ex-ante illustration**, not an investment recommendation or out-of-sample backtest.

**Exit ticket (discussion):** Could two investors with different risk aversions hold the exact same underlying risky-stock weights *in relative proportions*, while one lends and the other borrows? Under which assumptions would the answer change?

**References:** Markowitz (1952), *Portfolio Selection*, Journal of Finance; Tobin (1958), *Liquidity Preference as Behavior Towards Risk*, Review of Economic Studies; Sharpe (1966), *Mutual Fund Performance*, Journal of Business.

## Optional appendix — Analytical connection to mean–variance preferences

This section is **not needed for the main lesson**. It links the numerical experiment to the analytical Markowitz problem studied previously.

If $w$ contains the **total-wealth weights in the risky assets**, the risk-free weight is $w_f=1-\mathbf{1}^\top w$. With unrestricted risky weights, a common mean–variance objective is:

$$\max_w\; r_f+(\mu-r_f\mathbf{1})^\top w-\frac{\gamma}{2}w^\top\Sigma w,\qquad \gamma>0.$$

Its first-order condition gives

$$\Sigma w=\frac{1}{\gamma}(\mu-r_f\mathbf{1}),\qquad
w^*=\frac{1}{\gamma}\Sigma^{-1}(\mu-r_f\mathbf{1}),$$

provided $\Sigma$ is invertible and the solution is not subject to additional constraints. **Changing risk aversion $\gamma$ changes the scale of risky exposure, not its direction.** This is the analytical intuition for separation.

If the total risky allocation is positive, normalizing the direction gives the unconstrained risky tangency portfolio:

$$w_T=\frac{\Sigma^{-1}(\mu-r_f\mathbf{1})}{\mathbf{1}^\top\Sigma^{-1}(\mu-r_f\mathbf{1})}.$$

This expression should **not** be used as the solution when long-only, position limits, or gross-exposure constraints are imposed. In those cases use the constrained convex formulation above. Also note that the alternative convention $\max\{\mu_p-\lambda\sigma_p^2\}$ produces $1/(2\lambda)$ in the first-order solution — not $1/\lambda$.
